# Membership Inference Attack — ResNet-18 on CIFAR-10 (v2)


## Cell 1 — Setup

In [1]:
import os, csv, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms, models

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

for d in ['data','outputs/splits','outputs/checkpoints',
          'outputs/checkpoints/shadow_models',
          'outputs/checkpoints/attack_models','outputs/logs']:
    os.makedirs(d, exist_ok=True)

print('Directories ready.')

Device: cuda
GPU: Tesla T4
Directories ready.


## Cell 2 — Config

In [2]:
# ── Key changes from v1 ──────────────────────────────────────────
# TARGET_EPOCHS  : 40  → 80   more overfitting on 6K = bigger member gap
# SHADOW_EPOCHS  : 40  → 80   shadows must match target memorisation level
# NUM_SHADOWS    : 3   → 5    more shadow data = better attack classifier
# N_AUG_QUERIES  : 3   → 6    cleaner averaged confidence estimate
# ─────────────────────────────────────────────────────────────────
SEED            = 42
TARGET_EPOCHS   = 80
NUM_SHADOWS     = 5
SHADOW_EPOCHS   = 80
SHADOW_IN_SIZE  = 3000
SHADOW_OUT_SIZE = 3000
ATTACK_EPOCHS   = 80
NUM_CLASSES     = 10
BATCH_SIZE      = 128
N_AUG_QUERIES   = 6
FEATURE_DIM     = 23
MEMBER_THRESHOLD = np.log(2.0)   # corrected for 2:1 member/non-member imbalance

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print('Config:')
print(f'  Target epochs  : {TARGET_EPOCHS}')
print(f'  Shadow models  : {NUM_SHADOWS} x {SHADOW_EPOCHS} epochs')
print(f'  Aug queries    : {N_AUG_QUERIES} per sample')
print(f'  Feature dim    : {FEATURE_DIM}')

Config:
  Target epochs  : 80
  Shadow models  : 5 x 80 epochs
  Aug queries    : 6 per sample
  Feature dim    : 23


## Cell 3 — Models

In [3]:
def build_resnet18(num_classes=10):
    """CIFAR-adapted ResNet-18: 3x3 stem conv, no maxpool."""
    m = models.resnet18(weights=None, num_classes=num_classes)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    return m

class AttackMLP(nn.Module):
    """Binary classifier: 23-dim feature → member/non-member logit."""
    def __init__(self, in_dim=FEATURE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),     nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

n = sum(p.numel() for p in build_resnet18().parameters())
print(f'ResNet-18 parameters: {n:,}')

ResNet-18 parameters: 11,173,962


## Cell 4 — Data & Splits

In [4]:
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds_aug  = datasets.CIFAR10('data', train=True, download=True,  transform=train_tf)
train_ds_eval = datasets.CIFAR10('data', train=True, download=False, transform=eval_tf)

split_path = 'outputs/splits/cifar10_12k_seed42.npz'
if os.path.exists(split_path):
    sp          = np.load(split_path)
    target_idx  = sp['target_idx'].tolist()
    shadow_idx  = sp['shadow_idx'].tolist()
    holdout_idx = sp['holdout_idx'].tolist()
    print('Loaded existing splits.')
else:
    rng         = np.random.RandomState(42)
    all_idx     = rng.permutation(50000)[:12000]
    target_idx  = all_idx[:6000].tolist()
    shadow_idx  = all_idx[6000:9000].tolist()
    holdout_idx = all_idx[9000:12000].tolist()
    np.savez(split_path, target_idx=target_idx,
             shadow_idx=shadow_idx, holdout_idx=holdout_idx)
    print('Created new splits.')

print(f'Members: {len(target_idx):,} | Shadow reserved: {len(shadow_idx):,} | Holdout: {len(holdout_idx):,}')

100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


Created new splits.
Members: 6,000 | Shadow reserved: 3,000 | Holdout: 3,000


## Cell 5 — Train Target Model (~10 min)

In [5]:
@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return correct / total

def train_model(model, loader, epochs, lr=0.01, desc=''):
    opt   = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ce    = nn.CrossEntropyLoss()
    model.train()
    for ep in range(1, epochs+1):
        loss_sum, correct, total = 0.0, 0, 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            out  = model(x)
            loss = ce(out, y)
            loss.backward(); opt.step()
            loss_sum += loss.item() * x.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            total    += y.size(0)
        sched.step()
        if ep % 20 == 0 or ep == 1:
            print(f'  [{desc}] Epoch {ep:3d}/{epochs} | '
                  f'loss={loss_sum/total:.4f} | acc={correct/total:.4f}')
    return model

target_train_loader = DataLoader(Subset(train_ds_aug,  target_idx),
                                 batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
target_eval_loader  = DataLoader(Subset(train_ds_eval, target_idx),
                                 batch_size=256, shuffle=False, num_workers=2)
holdout_loader      = DataLoader(Subset(train_ds_eval, holdout_idx),
                                 batch_size=256, shuffle=False, num_workers=2)

print(f'Training ResNet-18 target for {TARGET_EPOCHS} epochs...')
t0     = time.time()
target = build_resnet18().to(DEVICE)
target = train_model(target, target_train_loader, TARGET_EPOCHS, desc='target')

mem_acc  = eval_acc(target, target_eval_loader)
hold_acc = eval_acc(target, holdout_loader)
gap      = mem_acc - hold_acc
print(f'\nMember acc : {mem_acc:.4f}')
print(f'Holdout acc: {hold_acc:.4f}')
print(f'Overfit gap: {gap:.4f}  ← bigger = easier for MIA')
print(f'Time: {(time.time()-t0)/60:.1f} min')

torch.save(target.state_dict(), 'outputs/checkpoints/best_baseline.pt')
print('Target saved.')

Training ResNet-18 target for 80 epochs...
  [target] Epoch   1/80 | loss=2.0688 | acc=0.2217
  [target] Epoch  20/80 | loss=0.6621 | acc=0.7617
  [target] Epoch  40/80 | loss=0.1777 | acc=0.9387
  [target] Epoch  60/80 | loss=0.0309 | acc=0.9925
  [target] Epoch  80/80 | loss=0.0185 | acc=0.9978

Member acc : 0.9995
Holdout acc: 0.7360
Overfit gap: 0.2635  ← bigger = easier for MIA
Time: 8.3 min
Target saved.


## Cell 6 — Feature Engineering

In [6]:
def make_features(conf, labels):
    """
    Build 23-dim attack feature vector:
      [0:10]  raw softmax confidence
      [10:20] sorted confidence (descending)
      [20]    Shannon entropy  — low = confident = likely member
      [21]    max confidence
      [22]    cross-entropy loss  — low = likely member (strongest signal)
    """
    entropy     = -(conf * (conf + 1e-9).log()).sum(1, keepdim=True)
    max_conf    = conf.max(1, keepdim=True).values
    sorted_conf = conf.sort(1, descending=True).values
    loss        = -conf[torch.arange(len(labels)), labels].log().unsqueeze(1)
    return torch.cat([conf, sorted_conf, entropy, max_conf, loss], dim=1)

@torch.no_grad()
def get_features(model, ds_aug, ds_eval, indices, n_aug=N_AUG_QUERIES):
    """
    Query model n_aug times with random augmentation per sample,
    average the confidence vectors, then build feature vectors.
    More aug passes = cleaner signal at the decision boundary.
    """
    idx    = list(indices)
    labels = torch.cat([lbl for _, lbl in
                        DataLoader(Subset(ds_eval, idx), batch_size=256,
                                   shuffle=False, num_workers=2)])
    model.eval()
    avg_conf = None
    for _ in range(n_aug):
        conf = torch.cat([
            torch.softmax(model(x.to(DEVICE)), 1).cpu()
            for x, _ in DataLoader(Subset(ds_aug, idx), batch_size=256,
                                   shuffle=False, num_workers=2)
        ])
        avg_conf = conf if avg_conf is None else avg_conf + conf
    avg_conf /= n_aug
    return make_features(avg_conf, labels), labels

print('Feature functions ready.')

Feature functions ready.


## Cell 7 — Train Shadow Models (~30 min)

In [7]:
sp   = np.load('outputs/splits/cifar10_12k_seed42.npz')
used = set(sp['target_idx'].tolist() +
           sp['shadow_idx'].tolist()  +
           sp['holdout_idx'].tolist())
pool = np.array([i for i in range(50000) if i not in used])
print(f'Shadow pool: {len(pool):,} samples (CIFAR-10 train minus fixed 12K split)')

rng = np.random.default_rng(SEED)
all_feats, all_labels, all_members = [], [], []

t0 = time.time()
for s in range(1, NUM_SHADOWS+1):
    print(f'\n── Shadow {s}/{NUM_SHADOWS} ──────────────────────────')
    perm    = rng.permutation(len(pool))
    in_idx  = pool[perm[:SHADOW_IN_SIZE]]           # 3K "members" for this shadow
    out_idx = pool[perm[SHADOW_IN_SIZE:SHADOW_IN_SIZE+SHADOW_OUT_SIZE]]  # 3K "non-members"

    loader = DataLoader(Subset(train_ds_aug, list(in_idx)),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    shadow = build_resnet18().to(DEVICE)
    shadow = train_model(shadow, loader, SHADOW_EPOCHS, desc=f'shadow-{s}')
    torch.save(shadow.state_dict(),
               f'outputs/checkpoints/shadow_models/shadow_{s}.pt')

    in_f,  in_l  = get_features(shadow, train_ds_aug, train_ds_eval, in_idx)
    out_f, out_l = get_features(shadow, train_ds_aug, train_ds_eval, out_idx)

    all_feats.append(torch.cat([in_f,  out_f]))
    all_labels.append(torch.cat([in_l, out_l]))
    all_members.append(torch.cat([
        torch.ones(len(in_idx),   dtype=torch.float32),
        torch.zeros(len(out_idx), dtype=torch.float32),
    ]))
    print(f'  Collected {len(in_idx):,} in + {len(out_idx):,} out')

feats   = torch.cat(all_feats)
labels  = torch.cat(all_labels)
members = torch.cat(all_members)
print(f'\nShadow training done in {(time.time()-t0)/60:.1f} min')
print(f'Attack training set: {len(members):,} samples '
      f'({int(members.sum()):,} in / {int((1-members).sum()):,} out)')

Shadow pool: 38,000 samples (CIFAR-10 train minus fixed 12K split)

── Shadow 1/5 ──────────────────────────
  [shadow-1] Epoch   1/80 | loss=2.2537 | acc=0.1597
  [shadow-1] Epoch  20/80 | loss=0.8459 | acc=0.6953
  [shadow-1] Epoch  40/80 | loss=0.1912 | acc=0.9370
  [shadow-1] Epoch  60/80 | loss=0.0349 | acc=0.9943
  [shadow-1] Epoch  80/80 | loss=0.0236 | acc=0.9970
  Collected 3,000 in + 3,000 out

── Shadow 2/5 ──────────────────────────
  [shadow-2] Epoch   1/80 | loss=2.2150 | acc=0.1737
  [shadow-2] Epoch  20/80 | loss=0.8467 | acc=0.7013
  [shadow-2] Epoch  40/80 | loss=0.1904 | acc=0.9380
  [shadow-2] Epoch  60/80 | loss=0.0368 | acc=0.9940
  [shadow-2] Epoch  80/80 | loss=0.0241 | acc=0.9967
  Collected 3,000 in + 3,000 out

── Shadow 3/5 ──────────────────────────
  [shadow-3] Epoch   1/80 | loss=2.2002 | acc=0.1817
  [shadow-3] Epoch  20/80 | loss=0.7921 | acc=0.7133
  [shadow-3] Epoch  40/80 | loss=0.2169 | acc=0.9273
  [shadow-3] Epoch  60/80 | loss=0.0361 | acc=0.9943

## Cell 8 — Train Attack Classifiers (~2 min)

In [8]:
attack_models = []
for c in range(NUM_CLASSES):
    mask = (labels == c)
    if mask.sum() < 10:
        attack_models.append(None); continue

    X, y    = feats[mask], members[mask]
    loader  = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
    m       = AttackMLP().to(DEVICE)
    opt     = optim.Adam(m.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()

    m.train()
    for _ in range(ATTACK_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss_fn(m(xb), yb).backward()
            opt.step()

    torch.save(m.state_dict(),
               f'outputs/checkpoints/attack_models/attack_class_{c}.pt')
    attack_models.append(m)
    print(f'  Class {c:2d}: {int(mask.sum()):,} samples '
          f'({int(y.sum()):,} in / {int((1-y).sum()):,} out)')

print('All 10 attack classifiers trained.')

  Class  0: 3,043 samples (1,523 in / 1,520 out)
  Class  1: 3,054 samples (1,530 in / 1,524 out)
  Class  2: 2,941 samples (1,459 in / 1,482 out)
  Class  3: 2,974 samples (1,483 in / 1,491 out)
  Class  4: 3,101 samples (1,483 in / 1,618 out)
  Class  5: 2,989 samples (1,528 in / 1,461 out)
  Class  6: 3,082 samples (1,532 in / 1,550 out)
  Class  7: 3,004 samples (1,519 in / 1,485 out)
  Class  8: 2,965 samples (1,492 in / 1,473 out)
  Class  9: 2,847 samples (1,451 in / 1,396 out)
All 10 attack classifiers trained.


## Cell 9 — Evaluate Attack

In [9]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

target.load_state_dict(torch.load('outputs/checkpoints/best_baseline.pt',
                                   map_location=DEVICE))
target.eval()

print('Extracting features from target model...')
mem_f,  mem_l  = get_features(target, train_ds_aug, train_ds_eval, target_idx)
non_f,  non_l  = get_features(target, train_ds_aug, train_ds_eval, holdout_idx)

all_f = torch.cat([mem_f, non_f])
all_l = torch.cat([mem_l, non_l])
all_m = torch.cat([
    torch.ones(len(target_idx),   dtype=torch.float32),
    torch.zeros(len(holdout_idx), dtype=torch.float32),
])

preds  = torch.zeros(len(all_m))
logits = torch.zeros(len(all_m))

for c in range(NUM_CLASSES):
    m = attack_models[c]
    if m is None: continue
    mask = (all_l == c)
    if mask.sum() == 0: continue
    with torch.no_grad():
        lg = m(all_f[mask].to(DEVICE)).cpu()
    logits[mask] = lg
    preds[mask]  = (lg > MEMBER_THRESHOLD).float()

y_true  = all_m.numpy()
y_pred  = preds.numpy()
y_score = torch.sigmoid(logits).numpy()

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred,    zero_division=0)
f1   = f1_score(y_true, y_pred,        zero_division=0)
auc  = roc_auc_score(y_true, y_score)
cm   = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
adv = 2.0 * (rec - fpr)

print('\n' + '='*52)
print('  Shadow Model MIA — ResNet-18 v2 Results')
print('='*52)
print(f'  Members / Non-members : {len(target_idx):,} / {len(holdout_idx):,}')
print(f'  Shadow models         : {NUM_SHADOWS} x {SHADOW_EPOCHS} epochs')
print(f'  Aug queries           : {N_AUG_QUERIES} per sample')
print('-'*52)
print(f'  Accuracy        : {acc:.4f}   (baseline = {len(target_idx)/len(all_m):.4f})')
print(f'  Precision       : {prec:.4f}')
print(f'  Recall (TPR)    : {rec:.4f}')
print(f'  False Pos. Rate : {fpr:.4f}')
print(f'  F1              : {f1:.4f}')
print(f'  AUC-ROC         : {auc:.4f}   (random = 0.5000)')
print(f'  Attacker Adv.   : {adv:.4f}   (random = 0.0000)')
print('-'*52)
print(f'  TN={tn:5d}  FP={fp:5d}')
print(f'  FN={fn:5d}  TP={tp:5d}')
print('='*52)

Extracting features from target model...

  Shadow Model MIA — ResNet-18 v2 Results
  Members / Non-members : 6,000 / 3,000
  Shadow models         : 5 x 80 epochs
  Aug queries           : 6 per sample
----------------------------------------------------
  Accuracy        : 0.7644   (baseline = 0.6667)
  Precision       : 0.8025
  Recall (TPR)    : 0.8578
  False Pos. Rate : 0.4223
  F1              : 0.8292
  AUC-ROC         : 0.7760   (random = 0.5000)
  Attacker Adv.   : 0.8710   (random = 0.0000)
----------------------------------------------------
  TN= 1733  FP= 1267
  FN=  853  TP= 5147


## Cell 10 — Save & Download Results

In [10]:
results = dict(
    attack='shadow_mia_resnet18_v2',
    target_epochs=TARGET_EPOCHS, num_shadows=NUM_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS, n_aug_queries=N_AUG_QUERIES,
    accuracy=round(acc,4), precision=round(prec,4),
    recall_tpr=round(rec,4), fpr=round(fpr,4),
    f1=round(f1,4), auc_roc=round(auc,4),
    attacker_advantage=round(adv,4),
    tn=tn, fp=fp, fn=fn, tp=tp,
)

with open('outputs/logs/shadow_mia_results_v2.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(results.keys()))
    w.writeheader(); w.writerow(results)

with open('outputs/logs/shadow_mia_per_sample_v2.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['true_member','predicted_member','attack_score','true_class'])
    for i in range(len(y_true)):
        w.writerow([int(y_true[i]), int(y_pred[i]),
                    f'{y_score[i]:.6f}', int(all_l[i].item())])

print('Results saved.')

from google.colab import files
files.download('outputs/logs/shadow_mia_results_v2.csv')
files.download('outputs/logs/shadow_mia_per_sample_v2.csv')
files.download('outputs/checkpoints/best_baseline.pt')
print('Downloads started.')

Results saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads started.
